# Efficient Attention: FlashAttention, Sparse and Linear Attention

চারটি অংশ, সবগুলোই Lesson 2-এর একই scaled dot-product attention সূত্র (`Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V`) থেকে কাজ করে, সাধারণ 2D `(T, d)` tensor-এর উপর প্রয়োগ করা — একটি sequence, একটি head — কারণ এখানকার প্রতিটি কৌশলের মূল বিষয় হলো COMPUTATION-এর `T`-নির্ভরতা, batch/multi-head হিসাবনিকাশ নয় যা Lesson 2 ইতিমধ্যে দেখিয়েছে।

0. **খরচ, concrete করা** — বাস্তব context দৈর্ঘ্যে `T x T` score matrix আসলে কত বড় হয়।

- **PART A. FlashAttention** — একটি EXACT পুনর্গঠন। `flash_attention_tiled` online-softmax recurrence implement করে, K/V-কে একবারে একটি block করে প্রক্রিয়া করে যাতে সম্পূর্ণ `T x T` score matrix কখনো materialize না হয়, এবং সংখ্যাগতভাবে যাচাই করা হয় যে এটি naive attention-এর সাথে IDENTICAL (floating point পর্যন্ত)। **লক্ষ্য করুন:** এটি সাধারণ PyTorch op-এর উপর একটি pure-Python loop, fused kernel নয় — এটি ALGORITHM-এর exactness ও এর কম peak-materialized-matrix আকার দেখায়, প্রকৃত GPU wall-clock speedup নয় (প্রকৃত FlashAttention-এর গতি আসে একটি fused CUDA kernel থেকে যা block-গুলোকে on-chip SRAM-এ ধরে রাখে; একই tensor op-এর উপর Python-স্তরের loop নিজস্ব overhead যোগ করে, এবং নিচে সরাসরি মাপা হিসেবে, প্রকৃত hardware-এ memory-efficient algorithm হওয়া সত্ত্বেও এখানে wall-clock-এ ধীর হয়)।

- **PART B. Sparse attention** — একটি প্রকৃত APPROXIMATION। প্রতিটি query-কে একটি local window ও কয়েকটি global token-এর মধ্যে সীমিত রাখে, ফলে গণনা করা (query, key) pair-এর সংখ্যা কমে। Part A-র বিপরীতে, এটি output বদলে দেয়।

- **PART C. Linear attention** — softmax-এর বদলে একটি kernel feature map, যা `softmax(Q K^T) V`-এর পরিবর্তে `phi(Q) @ (phi(K)^T @ V)` দিয়ে প্রকৃত `O(T)` compute দেয়। যাচাই করে যে causal recurrent (RNN-ধরনের, ধাপে ধাপে state update) রূপটি causal parallel (cumulative-sum) রূপের সাথে হুবহু মেলে, তারপর `T` বাড়ার সাথে non-causal parallel linear রূপের প্রকৃত wall-clock scaling naive attention-এর বিপরীতে মাপে — এটি সাধারণ PyTorch-এ সত্যিকারের, মাপযোগ্য গতি-সুবিধা, কারণ এতে মোট floating point operation কম এবং কোনো `T x T` matrix নেই; Part A-র মতো Python-loop-বনাম-fused-kernel তুলনা নয়।

**Runtime:** CPU-তে ~10-30 সেকেন্ড (Part C-এর scaling demo সবচেয়ে ধীর অংশ, `T=2048` পর্যন্ত)।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান।

In [ ]:
import math
import time

import torch
import torch.nn.functional as F

torch.manual_seed(0)

## 0. খরচ, concrete করা

Naive attention প্রতি head, প্রতি layer-এ একটি `(T, T)` score matrix materialize করে। এখানে float32-এ বাস্তবসম্মত 32টি head-এর জন্য একটি মাত্র layer-এ সেই matrix-এর আকার বিভিন্ন context দৈর্ঘ্যে হিসাব করা হয় — `T` বাড়লে খরচ `T^2` হারে বাড়ে।

In [ ]:
# ===========================================================================
# 0. খরচ, concrete করা
# ===========================================================================

def cost_demo():
    print("=" * 78)
    print("0. THE COST OF FULL ATTENTION, MADE CONCRETE")
    print("=" * 78)
    print("Naive attention materializes one (T, T) score matrix PER HEAD, per layer.")
    print("Sizes below are for that score matrix alone, float32 (4 bytes/element),")
    print("summed across a realistic 32 attention heads, for a SINGLE layer:\n")

    print(f"{'context length T':>18}{'one head (T*T)':>20}{'32 heads, 1 layer':>22}")
    for T in [4_096, 32_768, 131_072]:
        one_head_bytes = T * T * 4
        all_heads_bytes = one_head_bytes * 32
        print(f"{T:>18,}{one_head_bytes / 1e9:>17.2f} GB{all_heads_bytes / 1e9:>19.2f} GB")

    print("\n-> Quadrupling T (4,096 -> 131,072, a realistic jump for long-context")
    print("   models) grows the score matrix by 32x, not 4x -- T^2 growth. This is")
    print("   the exact O(T^2) cost flagged and deferred back in Phase 01 Lesson 5,")
    print("   section 5 (the trade-off: quadratic complexity). Phase 03 Lesson 6 and")
    print("   Phase 09 Lesson 2 each attack adjacent costs (how far a model can")
    print("   usefully attend at all; redundant recomputation during generation) --")
    print("   this lesson attacks the raw per-pass O(T^2) computation directly.")


cost_demo()

## 1. PART A: FlashAttention — exact, tiled, online-softmax attention

`naive_attention` হলো Lesson 2-এর অপরিবর্তিত সূত্র। `flash_attention_tiled` K/V-কে block ধরে প্রক্রিয়া করে, একটি চলমান max `m`, চলমান যোগফল `l` ও চলমান output accumulator `O` রাখে এবং প্রতিবার max বাড়লে `exp(old_max - new_max)` দিয়ে rescale করে। Demo-টি তিনটি জিনিস দেখায়: (১) বিভিন্ন `T`-তে (block_size-এর গুণিতক নয় এমনসহ) দুটি output হুবহু মেলে, (২) peak intermediate matrix-এর আকার `T*T` থেকে কমে `T*block_size` হয়, এবং (৩) একটি সৎ wall-clock তুলনা — pure-Python loop হওয়ায় এখানে tiled version আসলে ধীর।

In [ ]:
# ===========================================================================
# PART A: FLASHATTENTION -- EXACT, TILED, ONLINE-SOFTMAX ATTENTION
# ===========================================================================

def naive_attention(Q, K, V, causal=False):
    """Lesson 2 / Lesson 6-এর সূত্র, অপরিবর্তিত। Q, K: (T, d_k)। V: (T, d_v)।"""
    d_k = Q.shape[-1]
    T = Q.shape[-2]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)   # (T, T) -- যে matrix Part A materialize করা এড়ায়
    if causal:
        mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ V


def flash_attention_tiled(Q, K, V, block_size, causal=False):
    """Online-softmax algorithm (README §3): K/V-এর উপর `block_size` আকারের
    block-এ এগিয়ে যান, একটি চলমান max `m`, চলমান যোগফল `l`, এবং চলমান
    (unnormalized) output accumulator `O` বজায় রেখে; প্রতিবার কোনো নতুন block
    max বাড়ালে চলমান accumulator-গুলোকে exp(old_max - new_max) দিয়ে rescale
    করুন। সম্পূর্ণ (T, T) score matrix কখনো তৈরি হয় না -- এক সময়ে কেবল একটি
    (T, block_size) slice থাকে।

    প্রকৃত FlashAttention-এর তুলনায় সরলীকরণ: এটি কেবল K/V block-এর উপর tile
    করে (query দিক পুরোটাই থাকে)। প্রকৃত FlashAttention query dimension-ও tile
    করে যাতে প্রতিটি block pair একসাথে on-chip SRAM-এ ফিট করে -- কিন্তু
    online-softmax recurrence, যা আসল গাণিতিক কৌশল, উভয় ক্ষেত্রেই অভিন্ন।
    """
    T, d_k = Q.shape
    d_v = V.shape[-1]

    O = torch.zeros(T, d_v)
    l = torch.zeros(T, 1)                    # চলমান softmax denominator
    m = torch.full((T, 1), float("-inf"))    # চলমান row-wise max score

    num_blocks = math.ceil(T / block_size)
    q_idx = torch.arange(T).unsqueeze(1)     # (T, 1), নিচের causal mask-এর জন্য

    for j in range(num_blocks):
        k_start, k_end = j * block_size, min((j + 1) * block_size, T)
        K_j = K[k_start:k_end]               # (block, d_k) -- কেবল এই slice-টিই কখনো materialize হয়
        V_j = V[k_start:k_end]

        scores_j = Q @ K_j.transpose(-2, -1) / math.sqrt(d_k)   # (T, block)
        if causal:
            k_idx = torch.arange(k_start, k_end).unsqueeze(0)   # (1, block)
            scores_j = scores_j.masked_fill(k_idx > q_idx, float("-inf"))

        m_j = scores_j.max(dim=-1, keepdim=True).values         # (T, 1)
        m_new = torch.maximum(m, m_j)

        # এ পর্যন্ত জমা হওয়া সবকিছুর জন্য rescale factor। exp(-inf - -inf) = NaN
        # থেকে সুরক্ষিত, যা কেবল এমন একটি row-এর ক্ষেত্রে ঘটতে পারে যেটি এ পর্যন্ত
        # প্রতিটি block-এ শূন্যটি বৈধ key দেখেছে (এই loop-এর গঠন অনুযায়ী তা ঘটা
        # সম্ভব নয় -- README দেখুন -- তবে সতর্কতা হিসেবে guard রাখা হয়েছে)।
        alpha = torch.where(torch.isinf(m_new), torch.zeros_like(m_new), torch.exp(m - m_new))
        p_j = torch.exp(scores_j - m_new)                       # (T, block)

        l = alpha * l + p_j.sum(dim=-1, keepdim=True)
        O = alpha * O + p_j @ V_j
        m = m_new

    return O / l


def part_a_demo():
    print("\n" + "=" * 78)
    print("PART A: FLASHATTENTION -- TILED, ONLINE-SOFTMAX ATTENTION (EXACT)")
    print("=" * 78)

    d_k, d_v, block_size = 16, 16, 16
    print(f"Verifying flash_attention_tiled matches naive_attention EXACTLY")
    print(f"(block_size={block_size}), across several sequence lengths (including")
    print(f"lengths that are NOT multiples of block_size, to stress-test edges):\n")

    print(f"{'T':>8}{'causal':>10}{'max abs diff':>16}")
    for T in [8, 17, 64, 100]:
        for causal in [False, True]:
            Q = torch.randn(T, d_k)
            K = torch.randn(T, d_k)
            V = torch.randn(T, d_v)

            out_naive = naive_attention(Q, K, V, causal=causal)
            out_flash = flash_attention_tiled(Q, K, V, block_size, causal=causal)

            max_diff = (out_naive - out_flash).abs().max().item()
            print(f"{T:>8}{str(causal):>10}{max_diff:>16.2e}")
            assert torch.allclose(out_naive, out_flash, atol=1e-5), \
                f"Mismatch at T={T}, causal={causal}"

    print("\n-> Every max abs diff above is at floating-point precision (~1e-6 or")
    print("   smaller), not just 'close' -- flash_attention_tiled computes the")
    print("   mathematically EXACT same attention output as naive_attention, for")
    print("   every T tested including sizes that don't divide evenly by")
    print("   block_size. Zero quality tradeoff, by construction.")

    print("\n" + "-" * 78)
    print("Peak intermediate matrix size actually held in memory at once:")
    print("-" * 78)
    print(f"{'T':>10}{'naive (T x T)':>20}{'tiled (T x block)':>22}")
    for T in [1_024, 4_096, 16_384]:
        naive_elems = T * T
        tiled_elems = T * block_size
        print(f"{T:>10,}{naive_elems:>20,}{tiled_elems:>22,}")
    print(f"\n-> With block_size={block_size} fixed, the tiled approach's peak matrix")
    print(f"   size grows LINEARLY with T (T * {block_size}); naive's grows QUADRATICALLY")
    print(f"   (T * T). This is the real memory-shape saving FlashAttention's tiling")
    print(f"   is built on -- in a real fused GPU kernel, that smaller per-block")
    print(f"   matrix is what stays resident in fast on-chip SRAM instead of ever")
    print(f"   touching slow HBM, which is where FlashAttention's real wall-clock")
    print(f"   speedup on actual hardware comes from.")

    print("\n" + "-" * 78)
    print("Honest wall-clock check: naive vs. this PURE-PYTHON tiled loop")
    print("-" * 78)
    T, d = 2048, 32
    Q, K, V = torch.randn(T, d), torch.randn(T, d), torch.randn(T, d)
    with torch.no_grad():
        t0 = time.perf_counter()
        for _ in range(5):
            naive_attention(Q, K, V)
        naive_time = (time.perf_counter() - t0) / 5

        t0 = time.perf_counter()
        for _ in range(5):
            flash_attention_tiled(Q, K, V, block_size=128)
        tiled_time = (time.perf_counter() - t0) / 5

    print(f"naive_attention:        {naive_time * 1000:8.2f} ms  (one big matmul)")
    print(f"flash_attention_tiled:   {tiled_time * 1000:8.2f} ms  ({math.ceil(T / 128)} Python-level block iterations)")
    print(f"\n-> As documented at the top of this file: the tiled version is SLOWER")
    print(f"   here ({tiled_time / naive_time:.1f}x), not faster. That is expected and honest --")
    print(f"   this is a Python for-loop calling ordinary PyTorch ops once per block,")
    print(f"   which adds real per-iteration Python/dispatch overhead that a single")
    print(f"   large matmul doesn't pay. Real FlashAttention's speed advantage lives")
    print(f"   entirely inside a fused CUDA kernel that performs this exact same")
    print(f"   recurrence without ever leaving the GPU chip or paying Python overhead")
    print(f"   -- something no pure-Python loop over PyTorch tensor ops can reproduce.")
    print(f"   What this demo DOES prove for real: the algorithm is exact (above),")
    print(f"   and its peak materialized matrix shrinks from T*T to T*block_size.")


part_a_demo()

## 2. PART B: Sparse attention — local window + global token

`build_sparse_mask` একটি mask বানায় যেখানে প্রতিটি position কেবল নিজের আশেপাশের একটি নির্দিষ্ট window এবং প্রথম কয়েকটি "hub" (global) token দেখতে পারে (Longformer/BigBird-এর মতো)। Demo-টি দেখায় `T` বাড়লে গণনা করা pair-এর ভগ্নাংশ কমতে থাকে (খরচ `O(T * window)`), এবং Part A-র বিপরীতে এটি output সত্যিই বদলে দেয় — একটি প্রকৃত approximation।

In [ ]:
# ===========================================================================
# PART B: SPARSE ATTENTION -- LOCAL WINDOW + GLOBAL TOKENS
# ===========================================================================

def build_sparse_mask(T, window, num_global):
    """True = attend করার অনুমতি আছে। দুই ধরনের অনুমোদিত সংযোগ:
      - local: |i - j| <= window // 2 (একটি স্থির-আকারের neighborhood)
      - global: প্রথম `num_global`টি position প্রতিটি position-কে দেখতে পারে
        এবং প্রতিটি position তাদের দেখতে পারে -- অল্প কয়েকটি "hub" token যা
        তথ্যকে পুরো sequence জুড়ে ছড়িয়ে পড়তে দেয়, Longformer/BigBird-এ global
        token-গুলো ঠিক এই ভূমিকাই পালন করে।
    """
    i = torch.arange(T).unsqueeze(1)
    j = torch.arange(T).unsqueeze(0)
    local = (i - j).abs() <= window // 2
    is_global_row = i < num_global
    is_global_col = j < num_global
    return local | is_global_row | is_global_col


def part_b_demo():
    print("\n" + "=" * 78)
    print("PART B: SPARSE ATTENTION -- LOCAL WINDOW + GLOBAL TOKENS (APPROXIMATE)")
    print("=" * 78)
    print("The local-window piece here is exactly Phase 03 Lesson 6 section 3's")
    print("sliding-window attention; global tokens (a la Longformer/BigBird) are")
    print("added on top so information can still reach across the whole sequence")
    print("through a small number of always-visible hub positions.\n")

    print(f"{'T':>8}{'window':>10}{'num_global':>12}{'allowed pairs':>16}{'full T*T':>14}{'fraction computed':>20}")
    for T, window, num_global in [(256, 16, 2), (1024, 32, 4), (4096, 64, 8)]:
        mask = build_sparse_mask(T, window, num_global)
        allowed = mask.sum().item()
        full = T * T
        print(f"{T:>8}{window:>10}{num_global:>12}{allowed:>16,}{full:>14,}{allowed / full:>20.2%}")

    print("\n-> As T grows with window/num_global held fixed, the fraction of pairs")
    print("   actually computed keeps shrinking -- this mask's cost grows as")
    print("   O(T * window), linear in T, not O(T^2).")

    print("\n" + "-" * 78)
    print("Unlike Part A, this genuinely changes the output (a real approximation):")
    print("-" * 78)
    T, d_k, d_v, window, num_global = 256, 16, 16, 32, 4
    Q, K, V = torch.randn(T, d_k), torch.randn(T, d_k), torch.randn(T, d_v)

    out_full = naive_attention(Q, K, V)

    mask = build_sparse_mask(T, window, num_global)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    scores = scores.masked_fill(~mask, float("-inf"))
    out_sparse = F.softmax(scores, dim=-1) @ V

    mean_abs_diff = (out_full - out_sparse).abs().mean().item()
    are_close = torch.allclose(out_full, out_sparse, atol=1e-3)
    print(f"mean |full_attention_output - sparse_attention_output| = {mean_abs_diff:.4f}")
    print(f"outputs numerically equal (atol=1e-3)? {are_close}")
    print("\n-> A real, nonzero difference (and NOT numerically equal) -- sparse")
    print("   attention trades some of full attention's information (whatever a")
    print("   masked-out position could have contributed) for the O(T*window) cost")
    print("   above. Whether that trade is worth it depends entirely on how much a")
    print("   given task actually needs long-range, non-local, non-hub attention.")


part_b_demo()

## 3. PART C: Linear attention — kernel feature map, `O(T)` compute

Softmax-এর বদলে positive feature map `phi(x) = elu(x) + 1` ব্যবহার করে তিনটি রূপ সংজ্ঞায়িত করা হয়: non-causal parallel (`phi(Q) @ (phi(K)^T @ V)`, কোনো `(T, T)` matrix ছাড়াই), causal parallel (cumulative-sum, training-এর সময় ব্যবহৃত), এবং causal recurrent (ধাপে ধাপে একটি `(d_k, d_v)` state update, inference-এর সময় ব্যবহৃত constant-memory RNN-সমতুল্য রূপ)। এই cell-এর demo যাচাই করে যে recurrent ও parallel-cumsum রূপ সংখ্যাগতভাবে অভিন্ন।

In [ ]:
# ===========================================================================
# PART C: LINEAR ATTENTION -- KERNEL FEATURE MAP, O(T) COMPUTE
# ===========================================================================

def phi(x):
    """Positive elementwise feature map (Katharopoulos et al., 2020)।"""
    return F.elu(x) + 1.0


def linear_attention_parallel(Q, K, V):
    """Non-causal linear attention: phi(Q) @ (phi(K)^T @ V)। Matrix
    multiplication associative, তাই phi(K)^T @ V আগে গণনা করলে একটি
    (d_k, d_v) matrix পাওয়া যায় -- যা T-এর উপর নির্ভরশীল নয় -- এবং পুরো
    গণনার খরচ naive attention-এর O(T^2 * d)-এর বদলে O(T * d^2)। এখানে কোনো
    (T, T) matrix কখনোই তৈরি হয় না, Part A-র বিপরীতে (যা পুরোটা একসাথে
    materialize করা এড়ায় কিন্তু তবুও এর প্রতিটি T^2 entry গণনা ও স্পর্শ করে)
    -- এটি সত্যিই সেগুলো কখনো গণনা করে না।"""
    phi_Q, phi_K = phi(Q), phi(K)
    KV = phi_K.transpose(-2, -1) @ V                       # (d_k, d_v) -- (T, T) নয়
    Z = phi_K.sum(dim=-2)                                   # (d_k,) normalizer
    numerator = phi_Q @ KV                                  # (T, d_v)
    denominator = phi_Q @ Z.unsqueeze(-1)                    # (T, 1)
    return numerator / denominator


def linear_attention_causal_parallel_cumsum(Q, K, V):
    """Causal linear attention, PARALLEL (cumulative-sum) রূপ -- training-এর
    সময় ব্যবহৃত, যখন পুরো sequence একসাথে পাওয়া যায়। cumsum-এর মাধ্যমে চলমান
    outer product-এর একটি (T, d_k, d_v) tensor materialize করে; একটি ছোট demo-র
    জন্য ঠিক আছে, তবে লক্ষ্য করুন এটি constant-memory রূপ নয় (সেটি নিচের
    recurrent version) -- এটি তার training-time-parallel প্রতিরূপ।"""
    phi_Q, phi_K = phi(Q), phi(K)
    outer = phi_K.unsqueeze(-1) * V.unsqueeze(-2)            # (T, d_k, d_v), outer[t] = phi_K[t] (x) V[t]
    KV_cumulative = torch.cumsum(outer, dim=0)                # (T, d_k, d_v): KV_cumulative[t] = sum_{s<=t} outer[s]
    Z_cumulative = torch.cumsum(phi_K, dim=0)                  # (T, d_k)

    numerator = torch.einsum("td,tdv->tv", phi_Q, KV_cumulative)   # (T, d_v)
    denominator = (phi_Q * Z_cumulative).sum(dim=-1, keepdim=True)  # (T, 1)
    return numerator / denominator


def linear_attention_causal_recurrent(Q, K, V):
    """Causal linear attention, RECURRENT (ধাপে ধাপে) রূপ -- inference-এর সময়
    ব্যবহৃত constant-memory, RNN-সমতুল্য রূপ। একটি চলমান (d_k, d_v) state
    একবারে একটি token করে update হয়, ঠিক একটি RNN-এর hidden state-এর মতো --
    সাধারণ softmax attention-এর বিপরীতে (Phase 09 Lesson 2), আলাদা আলাদা
    token-এর কোনো ক্রমবর্ধমান KV cache-এর প্রয়োজনই নেই।"""
    T, d_k = Q.shape
    d_v = V.shape[-1]
    phi_Q, phi_K = phi(Q), phi(K)

    state = torch.zeros(d_k, d_v)     # চলমান (d_k, d_v) state -- একটি RNN-ধরনের hidden state
    z = torch.zeros(d_k)              # চলমান normalizer

    outputs = []
    for t in range(T):
        state = state + torch.outer(phi_K[t], V[t])    # state_t = state_{t-1} + phi(k_t) (x) v_t
        z = z + phi_K[t]
        out_t = (phi_Q[t] @ state) / (phi_Q[t] @ z)
        outputs.append(out_t)
    return torch.stack(outputs, dim=0)


def part_c_equivalence_demo():
    print("\n" + "=" * 78)
    print("PART C.1: LINEAR ATTENTION -- RECURRENT (RNN-style) == PARALLEL (cumsum)")
    print("=" * 78)

    T, d_k, d_v = 32, 8, 8
    Q, K, V = torch.randn(T, d_k), torch.randn(T, d_k), torch.randn(T, d_v)

    out_recurrent = linear_attention_causal_recurrent(Q, K, V)
    out_parallel = linear_attention_causal_parallel_cumsum(Q, K, V)
    max_diff = (out_recurrent - out_parallel).abs().max().item()

    print(f"T={T}: max abs diff between recurrent and parallel-cumsum forms: {max_diff:.2e}")
    assert torch.allclose(out_recurrent, out_parallel, atol=1e-5), "Recurrent/parallel forms disagree!"
    print("-> Numerically identical. The step-by-step state update really is just")
    print("   another way to compute the exact same cumulative sum -- causal linear")
    print("   attention IS an RNN, mathematically, not merely 'RNN-like.'")

    print("\nAlso confirming the NON-causal parallel form matches the causal one")
    print("with an all-True (no masking) equivalent -- i.e. it's the same")
    print("mechanism, just without the running/cumulative restriction to the past:")
    out_noncausal = linear_attention_parallel(Q, K, V)
    print(f"(non-causal) output shape: {tuple(out_noncausal.shape)}  "
          f"-- a DIFFERENT tensor from the causal outputs above by construction")
    print("(every position can see the whole sequence, not just positions <= t),")
    print("shown here only to confirm the function runs and shapes line up.")


part_c_equivalence_demo()

## 4. PART C.2: প্রকৃত wall-clock scaling — non-causal linear বনাম naive attention

এখানে দুই পক্ষই সাধারণ PyTorch matmul, কোনো পক্ষেই Python-স্তরের block loop নেই — তাই এটি একটি ন্যায্য তুলনা। বিভিন্ন `T`-তে সময় মাপা হয় এবং `T` দ্বিগুণ হলে সময় কত গুণ বাড়ে তা দেখানো হয়: naive-এর ক্ষেত্রে ~4x (`O(T^2)`), linear-এর ক্ষেত্রে ~2x (`O(T)`)-এর দিকে প্রবণতা থাকা উচিত।

In [ ]:
def part_c_scaling_demo():
    print("\n" + "=" * 78)
    print("PART C.2: REAL WALL-CLOCK SCALING -- NON-CAUSAL LINEAR vs. NAIVE ATTENTION")
    print("=" * 78)
    print("Unlike Part A's Python-loop-vs-single-matmul comparison, this one IS a")
    print("fair apples-to-apples comparison: both sides here are ordinary PyTorch")
    print("matmuls, no Python-level looping over blocks on either side. The only")
    print("difference is which matmuls get computed -- O(T^2 d) vs. O(T d^2).\n")

    d = 32
    Ts = [128, 256, 512, 1024, 2048]
    print(f"{'T':>8}{'naive (ms)':>14}{'linear (ms)':>14}{'speedup':>10}")
    naive_times, linear_times = [], []
    for T in Ts:
        Q, K, V = torch.randn(T, d), torch.randn(T, d), torch.randn(T, d)
        with torch.no_grad():
            t0 = time.perf_counter()
            for _ in range(10):
                naive_attention(Q, K, V)
            naive_t = (time.perf_counter() - t0) / 10

            t0 = time.perf_counter()
            for _ in range(10):
                linear_attention_parallel(Q, K, V)
            linear_t = (time.perf_counter() - t0) / 10

        naive_times.append(naive_t)
        linear_times.append(linear_t)
        print(f"{T:>8}{naive_t * 1000:>14.3f}{linear_t * 1000:>14.3f}{naive_t / linear_t:>9.1f}x")

    print("\nGrowth factor each time T doubles (naive should trend toward ~4x per")
    print("doubling -- O(T^2) -- linear should trend toward ~2x -- O(T)):")
    print(f"{'T -> 2T':>14}{'naive growth':>16}{'linear growth':>16}")
    for i in range(1, len(Ts)):
        naive_growth = naive_times[i] / naive_times[i - 1]
        linear_growth = linear_times[i] / linear_times[i - 1]
        print(f"{Ts[i-1]:>6} -> {Ts[i]:<5}{naive_growth:>16.2f}{linear_growth:>16.2f}")

    print("\n-> Timing noise means these growth factors won't land exactly on 4x/2x,")
    print("   but the TREND should be clearly visible: naive's per-doubling growth")
    print("   factor runs noticeably higher than linear attention's as T increases --")
    print("   the real, measurable consequence of O(T^2) vs. O(T) scaling, with no")
    print("   Python-loop overhead confound this time on either side.")


part_c_scaling_demo()

## সবগুলো demo একসাথে

`main()` উপরের সবগুলো demo এক সাথে ক্রমানুসারে চালায়। প্রতিটি demo ইতিমধ্যে নিজের cell-এ চলেছে, তাই দ্বিতীয়বার না চালানোর জন্য নিচে call-টি comment করা আছে।

In [ ]:
def main():
    cost_demo()
    part_a_demo()
    part_b_demo()
    part_c_equivalence_demo()
    part_c_scaling_demo()


# main()  # সব demo আবার একসাথে চালাতে uncomment করুন